In [ ]:
# ============================================================
# CELL 1 — Config
# ============================================================
VARIANTS = {
    "nano":  "RFDETRNano",
    "large": "RFDETRLarge",
}
PRECISIONS = ["int8"]
NUM_WARMUP = 5

# Real accuracy analysis uses a subset of COCO val2017 (official annotations),
# restricted to images containing at least one person.
NUM_CALIB_IMAGES = 50   # for NNCF int8 quantization calibration
NUM_BENCH_IMAGES = 50   # for latency benchmarking AND mAP accuracy eval


In [ ]:
# ============================================================
# CELL 2 — Install deps
# ============================================================
!pip install -q rfdetr openvino nncf pycocotools
!pip install -q onnx


In [ ]:
# ============================================================
# CELL 3 — Download a COCO val2017 subset (real ground truth, "person" only)
# ============================================================
import os
import zipfile
import requests
from pycocotools.coco import COCO

DATA_DIR = "/content/coco_val2017_subset"
os.makedirs(DATA_DIR, exist_ok=True)

ann_zip_path = "/content/annotations_trainval2017.zip"
ann_json_path = "/content/annotations/instances_val2017.json"

if not os.path.exists(ann_json_path):
    if not os.path.exists(ann_zip_path):
        print("Downloading COCO val2017 annotations (~240MB, one-time)...")
        url = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
        r = requests.get(url, stream=True, timeout=300)
        r.raise_for_status()
        with open(ann_zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print("Extracting instances_val2017.json...")
    with zipfile.ZipFile(ann_zip_path) as z:
        z.extract("annotations/instances_val2017.json", "/content")

coco_gt_full = COCO(ann_json_path)

person_cat_id = coco_gt_full.getCatIds(catNms=["person"])[0]  # official COCO id = 1
person_img_ids = sorted(coco_gt_full.getImgIds(catIds=[person_cat_id]))
person_img_ids = person_img_ids[: NUM_CALIB_IMAGES + NUM_BENCH_IMAGES]

calib_img_ids = person_img_ids[:NUM_CALIB_IMAGES]
bench_img_ids = person_img_ids[NUM_CALIB_IMAGES: NUM_CALIB_IMAGES + NUM_BENCH_IMAGES]

def download_images(img_ids, label):
    paths = []
    for iid in img_ids:
        meta = coco_gt_full.loadImgs(iid)[0]
        fname = meta["file_name"]
        path = os.path.join(DATA_DIR, fname)
        if not os.path.exists(path):
            url = f"http://images.cocodataset.org/val2017/{fname}"
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            with open(path, "wb") as f:
                f.write(r.content)
        paths.append(path)
    print(f"{label}: {len(paths)} images ready")
    return paths

calib_images = download_images(calib_img_ids, "calibration")
bench_images = download_images(bench_img_ids, "benchmark/accuracy")


loading annotations into memory...
Done (t=1.03s)
creating index...
index created!
calibration: 50 images ready
benchmark/accuracy: 50 images ready


In [ ]:
# ============================================================
# CELL 4 — Sanity check on the downloaded subset
# ============================================================
assert calib_images, "No calibration images found"
assert bench_images, "No benchmark images found"
print(f"{len(calib_images)} calibration images, {len(bench_images)} benchmark images")


50 calibration images, 50 benchmark images


In [ ]:
# ============================================================
# CELL 5 — Model constructors + fp32 ONNX export
# (needed as the source model for OpenVINO conversion)
# ============================================================
from rfdetr import RFDETRNano, RFDETRLarge

CTOR = {
    "nano":  RFDETRNano,
    "large": RFDETRLarge,
}

export_paths = {}          # (variant, "fp32") -> onnx path
resolutions = {}           # variant -> input resolution
class_names_by_variant = {}  # variant -> pretrained COCO class name list (no background slot)

import onnx

def onnx_input_hw(onnx_path):
    dims = onnx.load(onnx_path).graph.input[0].type.tensor_type.shape.dim
    return dims[2].dim_value, dims[3].dim_value

for variant, ctor in CTOR.items():
    if variant not in VARIANTS:
        continue

    model = ctor()

    # Capture the pretrained COCO class list before deleting the model.
    # We're using the model zero-shot, so it only knows COCO classes ('person', 'bicycle', ...).
    class_names = model.class_names
    assert "person" in class_names, f"'person' not found in class_names: {class_names}"
    class_names_by_variant[variant] = class_names

    out_dir = f"/content/export_{variant}"
    os.makedirs(out_dir, exist_ok=True)

    fp32_path = str(model.export(output_dir=out_dir, format="onnx"))
    export_paths[(variant, "fp32")] = fp32_path

    h, w = onnx_input_hw(fp32_path)
    resolutions[variant] = h

    del model

print(resolutions)


[2026-08-01 17:56:40] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


[2026-08-01 17:56:40] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-01 17:56:40] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-08-01 17:56:42] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


[2026-08-01 17:56:43] [WARNING] rf-detr - Pretrained weights at '/root/.roboflow/models/rf-detr-nano.pth' loaded only partially — this typically produces lower accuracy. 1 model parameter(s) not in checkpoint (left at random init): [_kp_active_mask]. Check that the model configuration (encoder, hidden_dim, out_feature_indexes, projector_scale, ...) matches the architecture the checkpoint was trained with.


[2026-08-01 17:56:43] [INFO] rf-detr - Exporting model to onnx format


...

[2026-08-01 17:56:47] [INFO] rf-detr - 
Successfully exported ONNX model: /content/export_nano/rfdetr-nano.onnx
[2026-08-01 17:56:47] [INFO] rf-detr - Successfully exported ONNX model to: /content/export_nano/rfdetr-nano.onnx
[2026-08-01 17:56:47] [INFO] rf-detr - Export completed successfully
[2026-08-01 17:56:48] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


[2026-08-01 17:56:48] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-08-01 17:56:48] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-08-01 17:56:49] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-large-2026.pth already exists with correct MD5 hash.


[2026-08-01 17:56:49] [WARNING] rf-detr - load_pretrain_weights: checkpoint lacks args.num_queries / args.group_detr; falling back to flat slice. With group_detr=13 this may scramble per-group query structure if the checkpoint was trained with group_detr > 1.
[2026-08-01 17:56:49] [WARNING] rf-detr - Pretrained weights at '/root/.roboflow/models/rf-detr-large-2026.pth' loaded only partially — this typically produces lower accuracy. 1 model parameter(s) not in checkpoint (left at random init): [_kp_active_mask]. Check that the model configuration (encoder, hidden_dim, out_feature_indexes, projector_scale, ...) matches the architecture the checkpoint was trained with.


[2026-08-01 17:56:49] [INFO] rf-detr - Exporting model to onnx format
[2026-08-01 17:56:55] [INFO] rf-detr - 
Successfully exported ONNX model: /content/export_large/rfdetr-large.onnx
[2026-08-01 17:56:55] [INFO] rf-detr - Successfully exported ONNX model to: /content/export_large/rfdetr-large.onnx
[2026-08-01 17:56:55] [INFO] rf-detr - Export completed successfully
{'nano': 384, 'large': 704}


In [ ]:
# ============================================================
# CELL 6 — OpenVINO IR conversion + NNCF INT8 quantization
# ============================================================
import openvino as ov
import nncf
import torch
from pathlib import Path
from torchvision import datasets, transforms

ov_core = ov.Core()
ov_export_paths = {}  # (variant, precision) -> .xml path

class ImagesOnlyFolder(torch.utils.data.Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = datasets.folder.default_loader(self.image_paths[idx])
        if self.transform:
            image = self.transform(image)
        return image

for variant in VARIANTS.keys():
    if (variant, "fp32") not in export_paths:
        continue

    fp32_onnx_path = export_paths[(variant, "fp32")]
    out_dir = f"/content/export_{variant}"
    size = resolutions[variant]

    # ONNX -> OpenVINO IR
    ov_model = ov.convert_model(fp32_onnx_path)
    ov_fp32_xml = os.path.join(out_dir, f"{variant}-ov-fp32.xml")
    ov.save_model(ov_model, ov_fp32_xml)
    ov_export_paths[(variant, "fp32")] = ov_fp32_xml

    if "int8" in PRECISIONS:
        prep_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((size, size)),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225]),
        ])
        val_dataset = ImagesOnlyFolder(calib_images, transform=prep_transform)
        dataset_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1)
        calibration_dataset = nncf.Dataset(dataset_loader)

        quantized_model = nncf.quantize(ov_model, calibration_dataset)

        ov_int8_xml = os.path.join(out_dir, f"{variant}-ov-int8.xml")
        ov.save_model(quantized_model, ov_int8_xml)
        ov_export_paths[(variant, "int8")] = ov_int8_xml

    print(f"{variant}: done ->", {k: v for k, v in ov_export_paths.items() if k[0] == variant})

Output()

Output()

nano: done -> {('nano', 'fp32'): '/content/export_nano/nano-ov-fp32.xml', ('nano', 'int8'): '/content/export_nano/nano-ov-int8.xml'}


Output()

Output()

large: done -> {('large', 'fp32'): '/content/export_large/large-ov-fp32.xml', ('large', 'int8'): '/content/export_large/large-ov-int8.xml'}


In [ ]:
# ============================================================
# CELL 7 — Benchmark with OpenVINO Core.compile_model
# ============================================================
import time
import numpy as np
import pandas as pd

ov_latency_rows = []

for (variant, precision), xml_path in ov_export_paths.items():
    if precision not in PRECISIONS and precision != "fp32":
        continue
    if not os.path.exists(xml_path):
        continue

    model = ov_core.read_model(xml_path)
    compiled = ov_core.compile_model(model, device_name="CPU")
    infer_request = compiled.create_infer_request()
    input_layer = compiled.input(0)

    size = resolutions[variant]
    prep_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((size, size)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])
    tensors = [prep_transform(datasets.folder.default_loader(p)).unsqueeze(0).numpy()
               for p in bench_images]

    for t in tensors[:NUM_WARMUP]:
        infer_request.infer({input_layer: t})

    times = []
    for t in tensors:
        start = time.perf_counter()
        infer_request.infer({input_layer: t})
        times.append(time.perf_counter() - start)

    times = np.array(times) * 1000
    bin_path = xml_path.replace(".xml", ".bin")
    size_mb = os.path.getsize(bin_path) / 1e6

    ov_latency_rows.append({
        "variant": variant, "precision": precision,
        "mean_latency_ms": times.mean(),
        "p50_latency_ms": np.percentile(times, 50),
        "p95_latency_ms": np.percentile(times, 95),
        "fps": 1000.0 / times.mean(),
        "model_size_mb": size_mb,
    })
    print(f"{variant:6s} {precision:5s}  mean={times.mean():6.2f} ms  fps={1000/times.mean():6.1f}  size={size_mb:6.1f} MB")

ov_latency_df = pd.DataFrame(ov_latency_rows)
ov_latency_df

nano   fp32   mean=318.84 ms  fps=   3.1  size=  53.8 MB
nano   int8   mean=271.24 ms  fps=   3.7  size=  28.5 MB
large  fp32   mean=1291.93 ms  fps=   0.8  size=  60.7 MB
large  int8   mean=1226.51 ms  fps=   0.8  size=  33.6 MB


,variant,precision,mean_latency_ms,p50_latency_ms,p95_latency_ms,fps,model_size_mb
0,nano,fp32,318.841204,297.167183,428.716444,3.136357,53.765954
1,nano,int8,271.244554,253.955899,341.499649,3.686710,28.489160
2,large,fp32,1291.929781,1169.116896,1804.095795,0.774036,60.713464
3,large,int8,1226.512002,1143.377399,1507.430455,0.815320,33.608232


In [ ]:
# ============================================================
# CELL 8 — Real accuracy evaluation (COCO mAP, "person", vs. official ground truth)
# ============================================================
import json as _json
import numpy as np
from PIL import Image
from pycocotools.cocoeval import COCOeval

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

COCO91_PERSON_IDX = 1  # official COCO "person" category id == the model's raw logit index

def split_outputs(raw_outputs):
    """Identify boxes vs. class logits by shape (boxes' last dim == 4),
    since OpenVINO doesn't guarantee output order/names after conversion."""
    a, b = raw_outputs
    if a.shape[-1] == 4:
        boxes, logits = a, b
    else:
        logits, boxes = a, b
    return logits, boxes

def get_person_index(logits_last_dim, variant):
    """RF-DETR's 91-wide raw logits index directly by official COCO id
    (0 = no-object, 1..90 = COCO ids, with 10 unused gap ids). Falls back
    to the contiguous class_names list if some export gives a different width."""
    if logits_last_dim == 91:
        return COCO91_PERSON_IDX
    names = class_names_by_variant[variant]
    if logits_last_dim == len(names):
        return names.index("person")
    raise ValueError(f"Unexpected logits width {logits_last_dim} for variant {variant}")

def run_accuracy_eval(compiled, variant, precision, score_thresh=0.3):
    input_layer = compiled.input(0)
    output_layers = compiled.outputs
    infer_request = compiled.create_infer_request()

    size = resolutions[variant]
    prep_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((size, size)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])

    coco_results = []

    for img_path, img_id in zip(bench_images, bench_img_ids):
        pil_img = Image.open(img_path).convert("RGB")
        orig_w, orig_h = pil_img.size

        tensor = prep_transform(pil_img).unsqueeze(0).numpy()
        infer_request.infer({input_layer: tensor})

        raw_outputs = [infer_request.get_tensor(o).data for o in output_layers]
        logits, boxes = split_outputs(raw_outputs)
        person_idx = get_person_index(logits.shape[-1], variant)

        scores_all = sigmoid(logits[0])
        person_scores = scores_all[:, person_idx]

        b = boxes[0]
        cx, cy, w, h = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
        x1 = (cx - w / 2) * orig_w
        y1 = (cy - h / 2) * orig_h
        x2 = (cx + w / 2) * orig_w
        y2 = (cy + h / 2) * orig_h

        keep = person_scores > score_thresh
        for xi1, yi1, xi2, yi2, sc in zip(x1[keep], y1[keep], x2[keep], y2[keep], person_scores[keep]):
            coco_results.append({
                "image_id": img_id,
                "category_id": person_cat_id,  # real official COCO person id
                "bbox": [float(xi1), float(yi1), float(xi2 - xi1), float(yi2 - yi1)],
                "score": float(sc),
            })

    if not coco_results:
        print(f"  [WARN] no detections above threshold for {variant} {precision} — mAP will be 0")
        return {"variant": variant, "precision": precision, "mAP": 0.0, "mAP50": 0.0}

    res_path = f"/content/{variant}_{precision}_person_results.json"
    with open(res_path, "w") as f:
        _json.dump(coco_results, f)

    coco_dt = coco_gt_full.loadRes(res_path)
    coco_eval = COCOeval(coco_gt_full, coco_dt, iouType="bbox")
    coco_eval.params.imgIds = bench_img_ids
    coco_eval.params.catIds = [person_cat_id]
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    return {
        "variant": variant, "precision": precision,
        "mAP": coco_eval.stats[0],     # AP @ IoU=0.50:0.95
        "mAP50": coco_eval.stats[1],   # AP @ IoU=0.50
    }


accuracy_rows = []
for (variant, precision), xml_path in ov_export_paths.items():
    if not os.path.exists(xml_path):
        continue
    model = ov_core.read_model(xml_path)
    compiled = ov_core.compile_model(model, device_name="CPU")
    row = run_accuracy_eval(compiled, variant, precision)
    accuracy_rows.append(row)
    print(row)

accuracy_df = pd.DataFrame(accuracy_rows)
accuracy_df


Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.05s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.542
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.742
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.590
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.224
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.671
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.864
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.191
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.518
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.572
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

,variant,precision,mAP,mAP50
0,nano,fp32,0.542137,0.741642
1,nano,int8,0.504592,0.733985
2,large,fp32,0.651993,0.850702
3,large,int8,0.578198,0.795162


In [ ]:
# ============================================================
# CELL 9 — Visual sanity check: show detections on a few benchmark images
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def run_detection(compiled, variant, image_path, score_thresh=0.5):
    input_layer = compiled.input(0)
    output_layers = compiled.outputs
    infer_request = compiled.create_infer_request()

    size = resolutions[variant]
    pil_img = Image.open(image_path).convert("RGB")
    orig_w, orig_h = pil_img.size

    prep_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize((size, size)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    tensor = prep_transform(pil_img).unsqueeze(0).numpy()
    infer_request.infer({input_layer: tensor})

    raw_outputs = [infer_request.get_tensor(o).data for o in output_layers]
    logits, boxes = split_outputs(raw_outputs)
    person_idx = get_person_index(logits.shape[-1], variant)

    scores_all = sigmoid(logits[0])
    person_scores = scores_all[:, person_idx]

    b = boxes[0]
    cx, cy, w, h = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    x1 = (cx - w / 2) * orig_w
    y1 = (cy - h / 2) * orig_h
    x2 = (cx + w / 2) * orig_w
    y2 = (cy + h / 2) * orig_h

    keep = person_scores > score_thresh
    return pil_img, x1[keep], y1[keep], x2[keep], y2[keep], person_scores[keep]


N_SHOW = 3
plot_precision = "int8" if any(p == "int8" for (_, p) in ov_export_paths) else "fp32"

for variant in VARIANTS:
    xml_path = ov_export_paths.get((variant, plot_precision))
    if xml_path is None or not os.path.exists(xml_path):
        continue

    model = ov_core.read_model(xml_path)
    compiled = ov_core.compile_model(model, device_name="CPU")

    fig, axes = plt.subplots(1, N_SHOW, figsize=(6 * N_SHOW, 6))
    fig.suptitle(f"{variant} ({plot_precision}) — person detections", fontsize=14)

    for ax, img_path in zip(axes, bench_images[:N_SHOW]):
        pil_img, x1, y1, x2, y2, scores = run_detection(compiled, variant, img_path)
        ax.imshow(pil_img)
        ax.set_title(f"{len(x1)} person(s) found")
        ax.axis("off")
        for xi1, yi1, xi2, yi2, sc in zip(x1, y1, x2, y2, scores):
            rect = patches.Rectangle((xi1, yi1), xi2 - xi1, yi2 - yi1,
                                      linewidth=2, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(xi1, max(yi1 - 5, 0), f"person {sc:.2f}", color="lime", fontsize=10,
                     bbox=dict(facecolor="black", alpha=0.5, pad=1))

    plt.tight_layout()
    plt.show()
